# Exploratory Data Analysis (EDA): PSO Stock Data
In this notebook, we perform a comprehensive Exploratory Data Analysis (EDA) on both the raw cleaned dataset and the engineered features dataset for PSO.
We will use **Matplotlib** and **Seaborn** to visualize stock trends, volume patterns, technical indicators, and feature correlations.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (14, 6)


## 1. Loading the Datasets
We will load both `pso_cleaned.csv` (the base OHLCV data) and `pso_features.csv` (the enriched dataset with technical indicators).


In [ ]:
ROOT_DIR = os.path.dirname(os.getcwd())
PROCESSED_DIR = os.path.join(ROOT_DIR, 'data', 'processed')

# Load raw cleaned data
df_clean = pd.read_csv(os.path.join(PROCESSED_DIR, 'pso_cleaned.csv'))
df_clean['date'] = pd.to_datetime(df_clean['date'])

# Load engineered features data
df_feat = pd.read_csv(os.path.join(PROCESSED_DIR, 'pso_features.csv'))
df_feat['date'] = pd.to_datetime(df_feat['date'])

print(f"Cleaned Data Shape: {df_clean.shape}")
print(f"Features Data Shape: {df_feat.shape}")


## 2. Analysis of the Base OHLCV Data
Let's first understand the overall price movement and trading volume.


In [ ]:
# Quick statistical summary
display(df_clean.describe())


In [ ]:
# Plotting the Closing Price History
plt.figure(figsize=(14, 6))
sns.lineplot(data=df_clean, x='date', y='close', color='royalblue')
plt.title('PSO - Historical Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing Price (Rs.)')
plt.show()


In [ ]:
# Plotting Trading Volume over time
plt.figure(figsize=(14, 5))
sns.lineplot(data=df_clean, x='date', y='volume', color='orange', alpha=0.7)
plt.fill_between(df_clean['date'], df_clean['volume'], color='orange', alpha=0.3)
plt.title('PSO - Trading Volume History')
plt.xlabel('Date')
plt.ylabel('Volume Traded')
plt.show()


## 3. Analysis of Engineered Features
We generated powerful technical indicators. Let's visualize them over the last year of trading to see how they behave.


In [ ]:
# Filter for the last 365 days for clearer technical indicator visualization
df_recent = df_feat.tail(365).copy()

# Moving Averages Plot
plt.figure(figsize=(14, 7))
plt.plot(df_recent['date'], df_recent['close'], label='Close Price', color='black', linewidth=2)
plt.plot(df_recent['date'], df_recent['sma_7'], label='SMA 7', color='blue', linestyle='--')
plt.plot(df_recent['date'], df_recent['sma_21'], label='SMA 21', color='green')
plt.plot(df_recent['date'], df_recent['sma_50'], label='SMA 50', color='red', linewidth=2)

plt.title('PSO - Close Price vs Simple Moving Averages (Last 1 Year)')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()


In [ ]:
# RSI (Relative Strength Index) Plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_recent['date'], df_recent['rsi_14'], color='purple', label='RSI (14)')

# Add overbought/oversold reference lines
ax.axhline(70, color='red', linestyle='--', alpha=0.5, label='Overbought (70)')
ax.axhline(30, color='green', linestyle='--', alpha=0.5, label='Oversold (30)')
ax.fill_between(df_recent['date'], y1=30, y2=70, color='gray', alpha=0.1)

ax.set_title('Relative Strength Index (RSI 14) - Momentum Indicator')
ax.set_ylabel('RSI')
ax.legend(loc='upper left')
plt.show()


In [ ]:
# MACD Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# MACD Lines
ax1.plot(df_recent['date'], df_recent['macd'], label='MACD Line', color='blue')
ax1.plot(df_recent['date'], df_recent['macd_signal'], label='Signal Line', color='orange')
ax1.set_title('MACD (Moving Average Convergence Divergence)')
ax1.set_ylabel('MACD Value')
ax1.legend()

# MACD Histogram
colors = ['green' if val >= 0 else 'red' for val in df_recent['macd_hist']]
ax2.bar(df_recent['date'], df_recent['macd_hist'], color=colors, alpha=0.6)
ax2.set_ylabel('Histogram')

plt.tight_layout()
plt.show()


## 4. Distribution and Correlations
Let's see how our features are correlated with each other, and the distribution of our target variable (`daily_return`).


In [ ]:
# Distribution of Daily Returns
plt.figure(figsize=(10, 5))
sns.histplot(df_feat['daily_return'].dropna(), bins=60, kde=True, color='teal')
plt.title('Distribution of Daily Percentage Returns')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
# Vertical line at 0
plt.axvline(0, color='black', linestyle='--', alpha=0.5)
plt.show()


In [ ]:
# Correlation Heatmap of Technical Features
cols_to_corr = ['close', 'volume', 'sma_7', 'sma_21', 'sma_50', 'rsi_14', 'macd', 'macd_hist', 'daily_return']
corr_matrix = df_feat[cols_to_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()
